## Asyncio functions

Although asyncio functions run and get results for many tasks, each function has specific functionality:

### asyncio.gather()
Returns a Future instance, allowing high level grouping of tasks:

In [ ]:
import asyncio
import random
from pprint import pprint
import nest_asyncio

nest_asyncio.apply()

async def coro(tag):
    print(">", tag)
    await asyncio.sleep(random.uniform(1, 3))
    print("<", tag)
    return tag


loop = asyncio.get_event_loop()

group1 = asyncio.gather(*[coro("group 1.{}".format(i)) for i in range(1, 6)])
group2 = asyncio.gather(*[coro("group 2.{}".format(i)) for i in range(1, 4)])
group3 = asyncio.gather(*[coro("group 3.{}".format(i)) for i in range(1, 10)])
print("group1", group1)
print("group2", group2)
print("group3", group3)
all_groups = asyncio.gather(group1, group2, group3)
print("all_groups", all_groups)
results = loop.run_until_complete(all_groups)
print("results", results)
loop.close()

pprint(results)

group1 <_GatheringFuture pending>
group2 <_GatheringFuture pending>
group3 <_GatheringFuture pending>
all_groups <_GatheringFuture pending>
> group 1.1
> group 1.2
> group 1.3
> group 1.4
> group 1.5
> group 2.1
> group 2.2
> group 2.3
> group 3.1
> group 3.2
> group 3.3
> group 3.4
> group 3.5
> group 3.6
> group 3.7
> group 3.8
> group 3.9
< group 3.7
< group 3.1
< group 1.4
< group 2.2
< group 3.5
< group 3.3
< group 2.1
< group 1.5
< group 3.4
< group 1.1
< group 1.3
< group 3.9
< group 3.8
< group 2.3
< group 3.6
< group 1.2
< group 3.2
results [['group 1.1', 'group 1.2', 'group 1.3', 'group 1.4', 'group 1.5'], ['group 2.1', 'group 2.2', 'group 2.3'], ['group 3.1', 'group 3.2', 'group 3.3', 'group 3.4', 'group 3.5', 'group 3.6', 'group 3.7', 'group 3.8', 'group 3.9']]


RuntimeError: Cannot close a running event loop

### asyncio.wait()

Supports waiting to be stopped after the first task is done, or after a specified timeout, allowing lower level precision of operations:

In [ ]:
import asyncio
import random
import nest_asyncio

nest_asyncio.apply()

async def coro(tag):
    print(">", tag)
    await asyncio.sleep(random.uniform(0.5, 5))
    print("<", tag)
    return tag

loop = asyncio.get_event_loop()

tasks = [coro(i) for i in range(1, 11)]

print("Get first result:")
finished, unfinished = loop.run_until_complete(
    asyncio.wait(tasks, return_when=asyncio.FIRST_COMPLETED))

for task in finished:
    print(task.result())
print("unfinished:", len(unfinished))

print("Get more results in 2 seconds:")
finished2, unfinished2 = loop.run_until_complete(
    asyncio.wait(unfinished, timeout=2))

for task in finished2:
    print(task.result())
print("unfinished2:", len(unfinished2))

print("Get all other results:")
finished3, unfinished3 = loop.run_until_complete(asyncio.wait(unfinished2))

for task in finished3:
    print(task.result())

loop.close()


### asyncio.wait vs asyncio.gather

asyncio.wait is more low level than asyncio.gather.

As the name suggests, asyncio.gather mainly focuses on gathering the results. It waits on a bunch of futures and returns their results in a given order.

asyncio.wait just waits on the futures. And instead of giving you the results directly, it gives done and pending tasks. You have to manually collect the values.

Moreover, you could specify to wait for all futures to finish or just the first one with wait.

## TaskGroup (Python 3.11+)
Update: Python 3.11 introduces TaskGroups which can "automatically" await more than one task without gather() or await():

In [ ]:
# Python 3.11+ ONLY!
async def main():
    async with asyncio.TaskGroup() as tg:
        task1 = tg.create_task(some_coro(...))
        task2 = tg.create_task(another_coro(...))
    print("Both tasks have completed now.")
